In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import create_extraction_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
import os

# Complete o nome da variavel antes de consultar os.environ.
if not os.environ.get("OPENAI_API_KEY"):
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("Chave da API OpenAI: ").strip()


In [ ]:
embeddings_model = OpenAIEmbeddings()
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=200)

In [ ]:
pdf_link = "etica-a-nicomaco.pdf"
loader = PyPDFLoader(pdf_link)
pages = loader.load_and_split()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4000,
    chunk_overlap = 0,
    length_function = len,
    add_start_index = True
)

chunks = text_splitter.split_documents(pages)

In [ ]:
db = Chroma.from_documents(chunks, embeddings_model,
persist_directory="db")

In [ ]:
vectordb = Chroma(persist_directory="db",
embedding_function=embeddings_model)
retriever = vectordb.as_retriever(search_kwargs={"k":3})

prompt = ChatPromptTemplate.from_template(
    """

    Responda a pergunta baseando-se APENAS no contexto abaixo.
    contexto: {context}
    Pergunta: {input}

    """

)

In [ ]:
from langchain_classic.chains import create_retrieval_chain

combine_docs_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [ ]:
def ask(question):
    response = retrieval_chain.invoke({"input": question})
    return response

In [ ]:
#INTERAÇÃO COM O USUÁRIO
user_input = input("Digite sua pergunta: ")
resultado = ask(user_input)

print("\nAnswer: ", resultado["answer"])
print("----------------------------")
print("Context (Primeiro documento recuperado): \n", resultado["context"][0].page_content)